In [1]:
from pathlib import Path
import pandas as pd

atl_path = Path("../data/raw/atl/atl_2025.csv")

atl = pd.read_csv(atl_path)

print(f"Raw ATL rows: {len(atl):,}")
print(f"Columns: {atl.columns.tolist()}")

Raw ATL rows: 17,592
Columns: ['publishTime', 'startTime', 'settlementDate', 'settlementPeriod', 'quantity']


In [2]:
atl["settlementDate"] = pd.to_datetime(atl["settlementDate"])

atl_2025 = atl[
    (atl["settlementDate"] >= "2025-01-01") &
    (atl["settlementDate"] <= "2025-12-31")
].copy()

print(f"Rows after date filtering: {len(atl_2025):,}")
print(
    f"Date range: "
    f"{atl_2025['settlementDate'].min().date()} "
    f"-> "
    f"{atl_2025['settlementDate'].max().date()}"
)

Rows after date filtering: 17,591
Date range: 2025-01-01 -> 2025-12-31


In [3]:
atl_2025["startTime"] = pd.to_datetime(
    atl_2025["startTime"],
    utc=True
)

print(
    f"Unique start times: "
    f"{atl_2025['startTime'].nunique():,}"
)

print(
    f"Minimum startTime: "
    f"{atl_2025['startTime'].min()}"
)

print(
    f"Maximum startTime: "
    f"{atl_2025['startTime'].max()}"
)

Unique start times: 17,491
Minimum startTime: 2025-01-01 00:00:00+00:00
Maximum startTime: 2025-12-31 23:30:00+00:00


In [4]:
duplicate_count = atl_2025.duplicated(
    subset=["startTime"]
).sum()

print(
    f"Duplicate startTime rows: "
    f"{duplicate_count:,}"
)

Duplicate startTime rows: 100


In [5]:
print("Missing values:")
print(atl_2025.isna().sum())

Missing values:
publishTime         0
startTime           0
settlementDate      0
settlementPeriod    0
quantity            0
dtype: int64


In [6]:
print("Demand summary:")
print(atl_2025["quantity"].describe())

Demand summary:
count    17591.000000
mean     33412.129725
std       7182.782744
min        634.000000
25%      28006.000000
50%      33624.000000
75%      38542.000000
max      53130.000000
Name: quantity, dtype: float64


In [7]:
timestamps = (
    atl_2025["startTime"]
    .drop_duplicates()
    .sort_values()
)

gaps = timestamps.diff().dropna()

gap_summary = (
    gaps.value_counts()
    .sort_index()
)

print(gap_summary)

startTime
0 days 00:30:00    17471
0 days 01:00:00       13
0 days 01:30:00        3
0 days 02:00:00        2
0 days 02:30:00        1
Name: count, dtype: int64


In [8]:
duplicate_rows = atl_2025[
    atl_2025.duplicated(
        subset=["startTime"],
        keep=False
    )
].sort_values(
    ["startTime", "publishTime"]
)

print(f"Duplicate rows: {len(duplicate_rows):,}")
print(
    f"Duplicate timestamp groups: "
    f"{duplicate_rows['startTime'].nunique():,}"
)

print(
    duplicate_rows[
        [
            "startTime",
            "publishTime",
            "settlementDate",
            "settlementPeriod",
            "quantity"
        ]
    ].head(30).to_string(index=False)
)

Duplicate rows: 198
Duplicate timestamp groups: 98
                startTime          publishTime settlementDate  settlementPeriod  quantity
2025-01-21 13:30:00+00:00 2025-01-21T14:10:08Z     2025-01-21                28   44010.0
2025-01-21 13:30:00+00:00 2025-01-21T14:25:06Z     2025-01-21                28   44351.0
2025-01-21 13:30:00+00:00 2025-01-21T15:40:11Z     2025-01-21                28   44386.0
2025-01-21 14:00:00+00:00 2025-01-21T14:55:06Z     2025-01-21                29   44176.0
2025-01-21 14:00:00+00:00 2025-01-21T15:40:12Z     2025-01-21                29   44210.0
2025-01-28 11:00:00+00:00 2025-01-28T11:55:07Z     2025-01-28                23   46687.0
2025-01-28 11:00:00+00:00 2025-01-28T13:10:07Z     2025-01-28                23   46711.0
2025-02-08 12:30:00+00:00 2025-02-08T13:25:05Z     2025-02-08                26    1361.0
2025-02-08 12:30:00+00:00 2025-02-08T16:10:17Z     2025-02-08                26   44767.0
2025-02-08 13:00:00+00:00 2025-02-08T13:55:04Z   

In [9]:
exact_duplicates = atl_2025[
    atl_2025.duplicated(
        keep=False
    )
]

print(
    f"Exact duplicate rows: "
    f"{len(exact_duplicates):,}"
)

Exact duplicate rows: 0


In [10]:
duplicate_summary = (
    duplicate_rows
    .groupby("startTime")
    .agg(
        observations=("startTime", "size"),
        publication_times=("publishTime", "nunique"),
        quantities=("quantity", "nunique"),
    )
)

print(duplicate_summary.head(20))

                           observations  publication_times  quantities
startTime                                                             
2025-01-21 13:30:00+00:00             3                  3           3
2025-01-21 14:00:00+00:00             2                  2           2
2025-01-28 11:00:00+00:00             2                  2           2
2025-02-08 12:30:00+00:00             2                  2           2
2025-02-08 13:00:00+00:00             2                  2           2
2025-02-08 13:30:00+00:00             2                  2           1
2025-02-08 16:30:00+00:00             2                  2           1
2025-02-15 09:00:00+00:00             2                  2           1
2025-02-15 09:30:00+00:00             2                  2           2
2025-02-15 10:00:00+00:00             2                  2           1
2025-03-11 16:00:00+00:00             2                  2           1
2025-03-13 10:00:00+00:00             2                  2           2
2025-0

In [11]:
minimum_demand = atl_2025.loc[
    atl_2025["quantity"].idxmin()
]

minimum_demand

publishTime              2025-02-08T17:40:07Z
startTime           2025-02-08 16:30:00+00:00
settlementDate            2025-02-08 00:00:00
settlementPeriod                           34
quantity                                634.0
Name: 1841, dtype: object

In [12]:
maximum_demand = atl_2025.loc[
    atl_2025["quantity"].idxmax()
]

maximum_demand

publishTime              2025-01-10T11:25:06Z
startTime           2025-01-10 10:30:00+00:00
settlementDate            2025-01-10 00:00:00
settlementPeriod                           22
quantity                              53130.0
Name: 556, dtype: object

In [13]:
atl_latest = (
    atl_2025
    .sort_values("publishTime")
    .drop_duplicates(
        subset=["startTime"],
        keep="last"
    )
)

print(
    f"Rows before: {len(atl_2025):,}"
)

print(
    f"Rows after latest-publication rule: "
    f"{len(atl_latest):,}"
)

print(
    f"Duplicate timestamps remaining: "
    f"{atl_latest.duplicated(subset=['startTime']).sum():,}"
)

Rows before: 17,591
Rows after latest-publication rule: 17,491
Duplicate timestamps remaining: 0


In [14]:
# Inspect the demand around the 634 MW observation

atl_latest = (
    atl_2025
    .sort_values("publishTime")
    .drop_duplicates(
        subset=["startTime"],
        keep="last"
    )
)

anomaly_time = pd.Timestamp(
    "2025-02-08 16:30:00+00:00"
)

window = atl_latest[
    (atl_latest["startTime"] >= anomaly_time - pd.Timedelta(hours=2)) &
    (atl_latest["startTime"] <= anomaly_time + pd.Timedelta(hours=2))
].sort_values("startTime")

window[
    [
        "startTime",
        "publishTime",
        "settlementDate",
        "settlementPeriod",
        "quantity"
    ]
]

,startTime,publishTime,settlementDate,settlementPeriod,quantity
1846,2025-02-08 14:30:00+00:00,2025-02-08T15:25:06Z,2025-02-08,30,44807.0
1845,2025-02-08 15:00:00+00:00,2025-02-08T15:55:05Z,2025-02-08,31,45091.0
1844,2025-02-08 15:30:00+00:00,2025-02-08T16:25:05Z,2025-02-08,32,45218.0
1843,2025-02-08 16:00:00+00:00,2025-04-28T10:40:11Z,2025-02-08,33,45758.0
1841,2025-02-08 16:30:00+00:00,2025-02-08T17:40:07Z,2025-02-08,34,634.0
1840,2025-02-08 17:00:00+00:00,2025-02-08T17:55:06Z,2025-02-08,35,46502.0
1839,2025-02-08 17:30:00+00:00,2025-02-08T18:25:06Z,2025-02-08,36,47132.0
1838,2025-02-08 18:00:00+00:00,2025-02-08T18:55:06Z,2025-02-08,37,46534.0
1837,2025-02-08 18:30:00+00:00,2025-02-08T19:25:06Z,2025-02-08,38,46211.0


In [15]:
expected_index = pd.date_range(
    start="2025-01-01 00:00:00+00:00",
    end="2025-12-31 23:30:00+00:00",
    freq="30min"
)

actual_index = pd.DatetimeIndex(
    atl_latest["startTime"].drop_duplicates().sort_values()
)

missing_times = expected_index.difference(actual_index)

print(f"Expected timestamps: {len(expected_index):,}")
print(f"Actual timestamps:   {len(actual_index):,}")
print(f"Missing timestamps:  {len(missing_times):,}")

print("\nMissing timestamps:")
for timestamp in missing_times:
    print(timestamp)

Expected timestamps: 17,520
Actual timestamps:   17,491
Missing timestamps:  29

Missing timestamps:
2025-01-21 11:30:00+00:00
2025-01-21 12:00:00+00:00
2025-01-31 05:30:00+00:00
2025-03-18 11:00:00+00:00
2025-03-18 11:30:00+00:00
2025-04-03 10:00:00+00:00
2025-04-03 19:30:00+00:00
2025-04-03 21:00:00+00:00
2025-04-03 21:30:00+00:00
2025-04-05 15:30:00+00:00
2025-04-05 16:00:00+00:00
2025-04-05 16:30:00+00:00
2025-04-05 22:30:00+00:00
2025-04-05 23:00:00+00:00
2025-04-05 23:30:00+00:00
2025-04-06 12:00:00+00:00
2025-04-06 12:30:00+00:00
2025-04-06 13:00:00+00:00
2025-04-06 13:30:00+00:00
2025-04-28 19:30:00+00:00
2025-04-29 13:30:00+00:00
2025-05-25 15:00:00+00:00
2025-05-29 10:00:00+00:00
2025-06-17 11:30:00+00:00
2025-09-06 22:30:00+00:00
2025-09-23 14:00:00+00:00
2025-10-14 22:30:00+00:00
2025-11-12 17:30:00+00:00
2025-11-18 11:30:00+00:00


In [16]:
# Show every available ATL version for the anomalous timestamp

anomaly = atl_2025[
    atl_2025["startTime"] == pd.Timestamp(
        "2025-02-08 16:30:00+00:00"
    )
].sort_values("publishTime")

anomaly[
    [
        "startTime",
        "publishTime",
        "settlementDate",
        "settlementPeriod",
        "quantity"
    ]
]

,startTime,publishTime,settlementDate,settlementPeriod,quantity
1842,2025-02-08 16:30:00+00:00,2025-02-08T17:25:07Z,2025-02-08,34,634.0
1841,2025-02-08 16:30:00+00:00,2025-02-08T17:40:07Z,2025-02-08,34,634.0


In [17]:
window = atl_latest[
    (atl_latest["startTime"] >= pd.Timestamp("2025-02-08 14:00:00+00:00")) &
    (atl_latest["startTime"] <= pd.Timestamp("2025-02-08 19:00:00+00:00"))
].sort_values("startTime")

window[
    [
        "startTime",
        "quantity"
    ]
].to_string(index=False)

'                startTime  quantity\n2025-02-08 14:00:00+00:00   45060.0\n2025-02-08 14:30:00+00:00   44807.0\n2025-02-08 15:00:00+00:00   45091.0\n2025-02-08 15:30:00+00:00   45218.0\n2025-02-08 16:00:00+00:00   45758.0\n2025-02-08 16:30:00+00:00     634.0\n2025-02-08 17:00:00+00:00   46502.0\n2025-02-08 17:30:00+00:00   47132.0\n2025-02-08 18:00:00+00:00   46534.0\n2025-02-08 18:30:00+00:00   46211.0\n2025-02-08 19:00:00+00:00   45405.0'

In [18]:
window = atl_latest[
    (atl_latest["startTime"] >= pd.Timestamp("2025-02-08 14:00:00+00:00")) &
    (atl_latest["startTime"] <= pd.Timestamp("2025-02-08 19:00:00+00:00"))
].sort_values("startTime")

print(
    window[
        ["startTime", "quantity"]
    ].to_string(index=False)
)

                startTime  quantity
2025-02-08 14:00:00+00:00   45060.0
2025-02-08 14:30:00+00:00   44807.0
2025-02-08 15:00:00+00:00   45091.0
2025-02-08 15:30:00+00:00   45218.0
2025-02-08 16:00:00+00:00   45758.0
2025-02-08 16:30:00+00:00     634.0
2025-02-08 17:00:00+00:00   46502.0
2025-02-08 17:30:00+00:00   47132.0
2025-02-08 18:00:00+00:00   46534.0
2025-02-08 18:30:00+00:00   46211.0
2025-02-08 19:00:00+00:00   45405.0


In [19]:
atl_clean = atl_latest.copy()

atl_clean = atl_clean.rename(
    columns={"quantity": "demand_mw"}
)

atl_clean["demand_quality_flag"] = "OK"

atl_clean.loc[
    atl_clean["startTime"] == pd.Timestamp(
        "2025-02-08 16:30:00+00:00"
    ),
    "demand_quality_flag"
] = "SOURCE_ANOMALY"

In [20]:
print(
    atl_clean["demand_quality_flag"].value_counts()
)

demand_quality_flag
OK                17490
SOURCE_ANOMALY        1
Name: count, dtype: int64


In [21]:
expected_index = pd.date_range(
    start="2025-01-01 00:00:00+00:00",
    end="2025-12-31 23:30:00+00:00",
    freq="30min"
)

actual_index = pd.DatetimeIndex(
    atl_clean["startTime"].sort_values()
)

missing_times = expected_index.difference(
    actual_index
)

missing_report = pd.DataFrame({
    "missing_startTime": missing_times
})

print(
    f"Missing timestamps: {len(missing_report):,}"
)

Missing timestamps: 29


In [22]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

atl_output = (
    processed_dir / "atl_2025_clean.parquet"
)

atl_clean.to_parquet(
    atl_output,
    index=False
)

print(f"Saved: {atl_output}")

Saved: ..\data\processed\atl_2025_clean.parquet


In [23]:
test_atl = pd.read_parquet(
    "../data/processed/atl_2025_clean.parquet"
)

print(f"Rows: {len(test_atl):,}")
print(
    "Duplicate startTime:",
    test_atl.duplicated(
        subset=["startTime"]
    ).sum()
)
print(
    "Quality flags:"
)
print(
    test_atl["demand_quality_flag"].value_counts()
)

Rows: 17,491
Duplicate startTime: 0
Quality flags:
demand_quality_flag
OK                17490
SOURCE_ANOMALY        1
Name: count, dtype: int64
